In [ ]:
import os, json, time, shutil, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
PROFILE = os.environ.get('NEURAL_DNA_PROFILE', 'local')
INPUT, OUTPUT = TASK / 'input' / PROFILE, TASK / 'output' / PROFILE
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
cfg = json.loads((INPUT / 'config.json').read_text())
SEED = cfg['seed']
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(cfg['threads'])

In [ ]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [ ]:
index = pd.read_csv(INPUT / 'prediction_index.csv')
truth = np.load(INPUT / 'truth.npz')['ancestry']
with np.load(INPUT / 'predictions.npz') as data:
    predictions = {key: data[key] for key in data.files}

In [ ]:
rows = []
for record in index.to_dict('records'):
    i, q = record['prediction_id'], record['query_id']
    target, called = truth[q], predictions['labels'][i]
    probability = predictions['probabilities'][i]
    rows.append({**record, 'block_accuracy': float(np.mean(called == target)),
        'boundary_f1': boundary_f1(target, called),
        'source_nll': float(-np.log(np.maximum(probability[np.arange(len(target)), target], 1e-300)).mean()),
        'brier': float(np.mean(np.sum((probability - np.eye(2)[target]) ** 2, axis=1))),
        'predicted_switches': int(np.sum(called[1:] != called[:-1])),
        'true_switches': int(np.sum(target[1:] != target[:-1]))})
trials = pd.DataFrame(rows)
trials.to_csv(OUTPUT / 'trial_metrics.csv', index=False)
summary = trials.groupby(['panel', 'markers', 'steps', 'kind', 'method'], as_index=False).agg(
    n=('mosaic_id', 'nunique'), block_accuracy=('block_accuracy', 'mean'), boundary_f1=('boundary_f1', 'mean'),
    source_nll=('source_nll', 'mean'), brier=('brier', 'mean'), perplexity=('perplexity', 'mean'),
    marker_relative_drift=('marker_relative_drift', 'mean'))
summary.to_csv(OUTPUT / 'summary.csv', index=False)

In [ ]:
paired = trials.pivot(index=['panel', 'markers', 'steps', 'kind', 'mosaic_id'], columns='method', values='block_accuracy')
paired['copying_minus_nearest'] = paired.copying - paired.nearest
paired.reset_index().to_csv(OUTPUT / 'paired_gains.csv', index=False)
# same model and two reference endpoints: convex-mixture hard calls equal nearest
exact = index.query("panel == 'exact' and method == 'nearest'").prediction_id.to_numpy()
mixture_calls = (predictions['mixture_weight_source_a'][exact] < 0.5).astype(int)
assert np.array_equal(mixture_calls, predictions['labels'][exact])
(OUTPUT / 'evaluation_checks.json').write_text(json.dumps({'two_source_mixture_matches_nearest': True}, indent=2))
display(summary)